# Norman 연구 결과 탐색

이 노트북은 커밋된 결과를 읽고, 인공 데이터로 공통 평가기의 입력과 출력을 점검합니다. 실행에 필요한 패키지는 Jupyter 및 NumPy입니다. 실제 학습과 비교표를 다시 만들려면 [01 논문 재현](01_paper_reproduction.md) 또는 [02 공통 튜닝](02_common_tuning.md)을 따르세요. [03 간 관련 적용](03_liver_application.md)에서 HCC 작업이 차단된 이유를 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import csv
import json
import math

root = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "manifests/common_norman_v1_s26.json").is_file()
)
tables = root / "reports/common/F/study/tables"

def read_tsv(path):
    with path.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream, delimiter="	"))

split = json.loads((root / "manifests/common_norman_v1_s26.json").read_text())
print(f"분할 {split['split_seed']}: {split['shared_genes']:,}개 유전자, "
      f"{split['evaluation_controls']}개 공통 평가 대조군")
print({role: counts["conditions"] for role, counts in split["roles"].items()})

## 논문 재현 수치

R1은 GEARS의 무변화 대비 정규화 MSE이고 R2는 scGPT의 변화량 Pearson입니다. 지표의 단위가 다르므로 두 행군의 숫자를 직접 비교하지 않습니다.

In [ ]:
paper = read_tsv(root / "reports/paper/reproduction_comparison.tsv")
for row in paper:
    author = float(row["author_value"])
    rerun = float(row["local_value"])
    diff = float(row["difference"])
    assert math.isclose(rerun - author, diff, abs_tol=1e-12)
    print(f"{row['track']} {row['group']}, {row['metric']}: "
          f"{author:.5f} → {rerun:.5f} ({diff:+.5f}, n={row['n_local_repetitions']})")

## 공통 기준선과 후보 확인

ΔRMSE가 작을수록 좋습니다. 적용 가능한 조합에 한정한 가산 기준선은 표의 별도 열을 참고하세요. 짝지은 개선량은 기준선 오차에서 후보 오차를 뺀 값입니다.

In [ ]:
for row in read_tsv(tables / "common_benchmark.tsv"):
    score = row["delta_rmse"]
    if score:
        print(f"{row['method']:>14}: {float(score):.5f} ({row['n_test_interventions']}개 개입)")
    else:
        print(f"{row['method']:>14}: {float(row['additive_subset_delta_rmse']):.5f} "
              f"({row['additive_subset_n']}개 조합만)")

for row in read_tsv(tables / "factorial_effects.tsv"):
    print(f"{row['contrast']}: {float(row['mean_gain']):+.5f} "
          f"[{float(row['condition_ci95_low']):+.5f}, {float(row['condition_ci95_high']):+.5f}]")

## 작은 입력으로 평가기 확인

임시 폴더의 인공 개입 한 개와 유전자 두 개를 실제 [공통 평가기](../src/common_eval.py)에 전달합니다. 완벽한 예측과 무변화 예측을 비교해 평가기가 동일한 대조군을 사용하고 예상한 ΔRMSE를 내는지 확인합니다.

In [ ]:
import sys
from tempfile import TemporaryDirectory
import numpy as np

sys.path.insert(0, str(root / "src"))
from common_eval import evaluate

with TemporaryDirectory() as tmp:
    example = Path(tmp)
    (example / "genes.json").write_text(json.dumps({"output": ["A", "B"]}))
    np.save(example / "eval_controls.npy", [[1.0, 2.0], [3.0, 4.0]])
    np.savez(example / "outcomes_val.npz", condition=["A"],
             expression_mean=[[4.0, 1.0]])
    exact = evaluate(example, "val", {"A": [4.0, 1.0]})
    unchanged = evaluate(example, "val", {"A": [2.0, 3.0]})
    assert exact["n_scored"] == 1 and exact["metrics"]["delta_rmse"] == 0.0
    assert math.isclose(exact["metrics"]["delta_pearson"], 1.0)
    assert unchanged["metrics"]["delta_rmse"] == 2.0
    assert unchanged["metrics"]["gain_vs_no_change"] == 0.0

print("Evaluator smoke passed: exact ΔRMSE=0; no-change ΔRMSE=2")